# Track 8 — Migración y Modernización Cloudera
**Rol:** CRB_MIGRACION | **Tiempo:** 15 min | **Criterio:** Assessment, conversión HiveQL, reconciliación, transferencia de conocimiento

In [ ]:
USE ROLE CRB_MIGRACION;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Evidencia: Catálogo de equivalencias listo

In [ ]:
-- Catálogo de equivalencias HiveQL → Snowflake
SELECT * FROM CREDIBANCO_HOL.MIGRACION.EQUIVALENCIAS_SQL
ORDER BY CATEGORIA;

In [ ]:
-- Filtrar por categoría: funciones
SELECT SENTENCIA_ORIGEN, SENTENCIA_SNOWFLAKE, NOTAS
FROM CREDIBANCO_HOL.MIGRACION.EQUIVALENCIAS_SQL
WHERE CATEGORIA = 'Funciones';

## Bloque 2 — Ejecutar: Reconciliación automática

In [ ]:
-- Reconciliar un día de datos migrados
CALL CREDIBANCO_HOL.MIGRACION.RECONCILIAR_DIA('2026-09-17');

In [ ]:
-- Ver resultado de la reconciliación
SELECT * FROM CREDIBANCO_HOL.MIGRACION.RECONCILIACION_DIARIA
ORDER BY FECHA DESC;

In [ ]:
-- Buscar en documentos de migración con Cortex Search
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'CREDIBANCO_HOL.MIGRACION.CS_MIGRACION_DOCS_USER',
    '{"query": "convertir HiveQL COLLECT_SET a Snowflake", "columns": ["CONTENIDO","TIPO"], "limit": 3}'
  )
);

## Bloque 3 — CoCo
Copia este prompt en Cortex Code:

> **Convierte este HiveQL a Snowflake SQL, explica cada cambio y genera un plan de migración por olas:**
> ```sql
> SELECT customer_id, COLLECT_SET(product) AS products,
>        SUM(amount) AS total
> FROM orders
> LATERAL VIEW EXPLODE(items) t AS item
> GROUP BY customer_id
> HAVING total > 1000000
> ```

In [ ]:
-- Verificación final
SELECT 'T8_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.MIGRACION.EQUIVALENCIAS_SQL) AS equivalencias,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.MIGRACION.RECONCILIACION_DIARIA) AS dias_reconciliados;